# **Ensenando a una computadora a diagnosticar enfermedades en tomates**

**Notebook practico** | Basado en: *Geetharamani & Arun Pandian (2019)*

---

Imagina que eres un agricultor y tienes miles de plantas de tomate. Un dia notas que algunas hojas tienen manchas extranas... pero hay tantas plantas que no puedes revisarlas todas. Seria genial tener un asistente que mire fotos de las hojas y te diga exactamente que enfermedad tienen, no?

Eso es justo lo que vamos a construir aqui: **una inteligencia artificial que aprende a reconocer 10 enfermedades del tomate mirando miles de fotos de hojas**. Y lo mejor es que lo vamos a hacer de dos formas distintas para comparar cual funciona mejor.

**Antes de empezar:** Asegurate de tener la GPU activada en Colab (es como ponerle un motor turbo a la computadora). Ve a `Entorno de ejecucion > Cambiar tipo de entorno > GPU T4`.

**Que vamos a hacer:**

| Paso | Que haremos | Tiempo aprox. |
|------|-------------|---------------|
| 1 | Preparar las herramientas | 1 min |
| 2 | Descargar fotos de hojas de tomate | 2 min |
| 3 | Preparar las fotos para que el modelo las entienda | 1 min |
| 4 | Construir nuestro primer modelo (CNN desde cero) | 1 min |
| 5 | Entrenar el modelo — aqui la computadora aprende! | ~10-15 min |
| 6 | Probar un atajo: usar un modelo que ya sabe "ver" (Transfer Learning) | ~5 min |
| 7 | Comparar: quien aprendio mejor? | 1 min |
| 8 | Reflexiones y proximos pasos | - |

---
## 1. Preparando las herramientas

Antes de cocinar hay que sacar los ingredientes, no? Aqui es lo mismo: necesitamos cargar las herramientas (librerias) que usaremos para procesar imagenes, construir la red neuronal y visualizar resultados.

Solo dale "play" a las siguientes dos celdas y espera a que digan "Entorno listo".

In [ ]:
# ==============================================================
# INSTALACION DE DEPENDENCIAS
# kagglehub: permite descargar datasets de Kaggle sin configurar API keys
# (torch y torchvision ya vienen preinstalados en Colab)
# ==============================================================
!pip install -q kagglehub

In [ ]:
# ==============================================================
# IMPORTS Y CONFIGURACION DEL ENTORNO
# ==============================================================
import os
import time
import random
import shutil
from pathlib import Path
from collections import Counter

# --- Computacion numerica y visualizacion ---
import numpy as np                          # operaciones con arrays
import matplotlib.pyplot as plt             # graficas
from PIL import Image                       # lectura de imagenes

# --- PyTorch: framework de deep learning ---
import torch                                # tensores y autograd
import torch.nn as nn                       # capas y modelos
import torch.optim as optim                 # optimizadores (Adam, SGD)
from torch.utils.data import DataLoader, Subset  # carga de datos en batches

# --- Torchvision: utilidades para vision por computadora ---
import torchvision.transforms as T          # transformaciones de imagenes
from torchvision.datasets import ImageFolder # carga datasets organizados en carpetas
from torchvision import models              # modelos preentrenados (ResNet, etc.)

# --- Metricas ---
from sklearn.metrics import confusion_matrix, classification_report

# ==============================================================
# REPRODUCIBILIDAD: fijar semillas para obtener resultados consistentes
# ==============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Desactivar algoritmos no deterministas para reproducibilidad
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==============================================================
# DETECCION DE GPU
# En Colab con GPU habilitada, device sera 'cuda'
# Si no hay GPU, se usara CPU (mucho mas lento)
# ==============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ADVERTENCIA: No se detecto GPU. El entrenamiento sera muy lento.')
    print('Activa la GPU en: Entorno de ejecucion > Cambiar tipo de entorno > GPU T4')

# Configuracion de graficas
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('\nEntorno listo.')

---
## 2. Consiguiendo las fotos de hojas de tomate

Nuestro modelo va a aprender mirando ejemplos, igual que nosotros. Si quieres aprender a distinguir un perro de un gato, necesitas ver muchos perros y muchos gatos. Aqui necesitamos miles de fotos de hojas — sanas y enfermas.

Usaremos el dataset **PlantVillage**, una coleccion publica creada por investigadores para que cualquiera pueda experimentar con inteligencia artificial aplicada a la agricultura. Contiene mas de 54,000 fotos de hojas de distintas plantas.

Nosotros nos quedaremos solo con las **10 clases de tomate** (~18,000 fotos):

| # | Clase | En palabras simples |
|---|-------|---------------------|
| 1 | Tomato_healthy | Hoja sana (nuestro "control") |
| 2 | Tomato_Bacterial_spot | Manchas causadas por bacterias |
| 3 | Tomato_Early_blight | Tizon temprano (hongo que aparece pronto) |
| 4 | Tomato_Late_blight | Tizon tardio (el que destruyo los cultivos de papa en Irlanda en 1845!) |
| 5 | Tomato_Leaf_Mold | Moho en la hoja |
| 6 | Tomato_Septoria_leaf_spot | Manchitas causadas por el hongo Septoria |
| 7 | Tomato_Spider_mites | Dano por arana roja (un acaro diminuto) |
| 8 | Tomato_Target_Spot | Manchas en forma de diana |
| 9 | Tomato_mosaic_virus | Virus del mosaico (hace que la hoja se vea "arrugada") |
| 10 | Tomato_Yellow_Leaf_Curl_Virus | Virus que enrolla y amarillea las hojas |

Pregunta para pensar: Si solo tenemos fotos tomadas en laboratorio con fondo uniforme, crees que el modelo funcionaria igual de bien con fotos tomadas en el campo real? Esa es una de las grandes preguntas abiertas en este tema.

In [ ]:
# ==============================================================
# DESCARGA DEL DATASET DESDE KAGGLE
# kagglehub descarga automaticamente sin necesidad de API key
# ==============================================================
import kagglehub

print('Descargando dataset PlantVillage...')
dataset_path = kagglehub.dataset_download('abdallahalidev/plantvillage-dataset')
print(f'Dataset descargado en: {dataset_path}')

# Buscar la carpeta que contiene las imagenes organizadas por clase
# La estructura tipica es: .../plantvillage-dataset/versions/X/color/
# o .../plantvillage-dataset/versions/X/plantvillage dataset/color/
possible_roots = [
    Path(dataset_path) / 'color',
    Path(dataset_path) / 'plantvillage dataset' / 'color',
    Path(dataset_path) / 'PlantVillage',
    Path(dataset_path),
]

dataset_root = None
for p in possible_roots:
    if p.exists():
        subdirs = [d for d in p.iterdir() if d.is_dir()]
        # Verificar que contiene carpetas con nombres de tomate
        tomato_dirs = [d for d in subdirs if 'tomato' in d.name.lower() or 'Tomato' in d.name]
        if tomato_dirs:
            dataset_root = p
            break

if dataset_root is None:
    # Buscar recursivamente
    for root, dirs, files in os.walk(dataset_path):
        tomato_dirs = [d for d in dirs if 'tomato' in d.lower()]
        if len(tomato_dirs) >= 5:
            dataset_root = Path(root)
            break

assert dataset_root is not None, 'No se encontro el directorio del dataset. Revisa la estructura.'
print(f'Raiz del dataset: {dataset_root}')
print(f'Carpetas encontradas: {len(list(dataset_root.iterdir()))}')

In [ ]:
# ==============================================================
# FILTRAR SOLO LAS 10 CLASES DE TOMATE
# Copiamos las carpetas de tomate a un directorio temporal
# para que ImageFolder las cargue correctamente
# ==============================================================

# Buscar todas las carpetas que contengan 'Tomato' o 'tomato'
all_dirs = sorted([d for d in dataset_root.iterdir() if d.is_dir()])
tomato_dirs = [d for d in all_dirs if 'tomato' in d.name.lower()]

print(f'Clases de tomate encontradas ({len(tomato_dirs)}):')
for d in tomato_dirs:
    n_imgs = len([f for f in d.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    print(f'  {d.name}: {n_imgs} imagenes')

# Crear directorio filtrado con solo las clases de tomate
TOMATO_DIR = Path('/tmp/tomato_dataset')
if TOMATO_DIR.exists():
    shutil.rmtree(TOMATO_DIR)
TOMATO_DIR.mkdir(parents=True)

total_imgs = 0
for d in tomato_dirs:
    dest = TOMATO_DIR / d.name
    # Usar symlinks para ahorrar espacio y tiempo
    os.symlink(str(d), str(dest))
    n = len([f for f in d.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    total_imgs += n

print(f'\nTotal: {total_imgs} imagenes de tomate en {len(tomato_dirs)} clases')

# --- Visualizar ejemplos de cada clase ---
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for idx, d in enumerate(sorted(TOMATO_DIR.iterdir())):
    imgs = sorted([f for f in d.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    if imgs:
        img = Image.open(imgs[0])
        axes[idx].imshow(img)
    # Nombre corto: quitar prefijo 'Tomato___' o 'Tomato_'
    nombre = d.name.replace('Tomato___', '').replace('Tomato_', '').replace('_', ' ')
    axes[idx].set_title(nombre, fontsize=9, fontweight='bold')
    axes[idx].axis('off')

fig.suptitle('Ejemplo de cada clase de tomate', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Distribucion de imagenes por clase ---
nombres = []
cantidades = []
for d in sorted(TOMATO_DIR.iterdir()):
    nombre = d.name.replace('Tomato___', '').replace('Tomato_', '').replace('_', ' ')
    n = len([f for f in d.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    nombres.append(nombre)
    cantidades.append(n)

fig, ax = plt.subplots(figsize=(12, 5))
colores = plt.cm.viridis(np.linspace(0.2, 0.8, len(nombres)))
bars = ax.barh(nombres, cantidades, color=colores, edgecolor='white')
for bar, val in zip(bars, cantidades):
    ax.text(val + 20, bar.get_y() + bar.get_height()/2, str(val),
            va='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Numero de imagenes', fontsize=11)
ax.set_title('Distribucion de imagenes por clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Preparando las fotos para el modelo

Las fotos originales vienen en distintos tamanos y con valores de color "crudos". Antes de darselas al modelo, necesitamos estandarizarlas — algo asi como ponerlas todas en el mismo formato para que el modelo no se confunda.

Que vamos a hacer con cada foto:

1. **Redimensionar a 128x128 pixeles** — Todas del mismo tamano, como en el paper original. Es un buen balance: suficiente detalle para ver las manchas, pero no tanto como para que la computadora se tarde una eternidad.

2. **Convertir a numeros** — Recordemos que para la computadora una imagen es solo una tabla de numeros (pixeles).

3. **Normalizar** — Ajustar los valores para que esten en un rango "comodo" para la red neuronal. Es como cuando conviertes grados Fahrenheit a Celsius: los datos son los mismos, pero en una escala mas facil de trabajar.

Ademas, vamos a separar las fotos en dos grupos:
- **80% para entrenar** — con estas el modelo aprende.
- **20% para validar** — con estas comprobamos si realmente aprendio o solo "memorizo" las respuestas (como cuando estudias para un examen vs. cuando te evaluan con preguntas nuevas).

In [ ]:
# ==============================================================
# TRANSFORMACIONES DE IMAGENES
# ==============================================================

# Transformaciones para entrenamiento
# - Resize(128): redimensionar a 128x128 pixeles (como el paper)
# - ToTensor: convertir imagen PIL a tensor PyTorch (valores 0-1)
# - Normalize: centrar valores usando media y desviacion estandar de ImageNet
#   Esto ayuda a que el modelo converja mas rapido
train_transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],   # media de ImageNet
                std=[0.229, 0.224, 0.225]),     # desv. estandar de ImageNet
])

# Para validacion usamos las mismas transformaciones (sin data augmentation)
val_transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

In [ ]:
# ==============================================================
# SPLIT TRAIN/VAL Y DATALOADERS
# ==============================================================

# ImageFolder carga automaticamente las imagenes organizadas en carpetas
# Cada subcarpeta se interpreta como una clase
full_dataset = ImageFolder(str(TOMATO_DIR), transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)

print(f'Clases detectadas ({num_classes}):')
for i, name in enumerate(class_names):
    print(f'  {i}: {name}')

# Split 80% train / 20% validacion (estratificado)
# Usamos indices para poder aplicar transformaciones diferentes
n_total = len(full_dataset)
indices = list(range(n_total))
np.random.shuffle(indices)

split = int(0.8 * n_total)
train_indices = indices[:split]
val_indices = indices[split:]

print(f'\nTotal de imagenes: {n_total}')
print(f'Entrenamiento: {len(train_indices)} ({len(train_indices)/n_total:.0%})')
print(f'Validacion: {len(val_indices)} ({len(val_indices)/n_total:.0%})')

# Crear subsets
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

# Para validacion, creamos otro dataset con val_transform
val_dataset_proper = ImageFolder(str(TOMATO_DIR), transform=val_transform)
val_dataset = Subset(val_dataset_proper, val_indices)

# DataLoaders: cargan los datos en batches para el entrenamiento
# - batch_size=128: procesar 128 imagenes a la vez (como el paper)
# - shuffle=True en train: mezclar datos cada epoca para mejor generalizacion
# - num_workers=2: hilos paralelos para cargar datos mas rapido
# - pin_memory=True: acelera la transferencia CPU->GPU
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)

# Verificar un batch
images, labels = next(iter(train_loader))
print(f'\nBatch de ejemplo:')
print(f'  Imagenes: {images.shape}  (batch, canales, alto, ancho)')
print(f'  Etiquetas: {labels.shape}  (batch,)')

---
## 4. Construyendo nuestro primer modelo: la CNN del paper

Ahora viene lo emocionante: vamos a construir la red neuronal que "ve" las hojas.

Si viste el taller teorico, recordaras que una CNN funciona como un sistema de filtros en cadena: las primeras capas detectan cosas simples (bordes, cambios de color) y las capas mas profundas combinan esas pistas para reconocer patrones complejos (manchas, texturas de enfermedad, formas de hojas).

Nuestra CNN tiene **6 bloques de "observacion"** (convolucionales), cada uno con:
- Un **filtro** que busca patrones en la imagen
- **Normalizacion** para que el aprendizaje sea estable
- **ReLU** — una regla simple: si el valor es negativo, ponlo en cero (esto le da al modelo la capacidad de aprender relaciones no lineales)
- **Max Pooling** — se queda solo con lo mas importante y reduce el tamano de la imagen a la mitad

Despues, dos **capas densas** toman todas esas caracteristicas y deciden: "esta hoja tiene tizon tardio" o "esta hoja esta sana".

La imagen va reduciendo su tamano en cada paso:
`128 → 64 → 32 → 16 → 8 → 4 → 2 pixeles`

Pero al mismo tiempo, el modelo va "entendiendo" mas:
`3 filtros → 32 → 64 → 128 → 256 → 512 filtros`

Es como si el modelo fuera haciendo un resumen cada vez mas abstracto de lo que ve.

In [ ]:
# ==============================================================
# MODELO CNN DEL PAPER (adaptado a 10 clases de tomate)
# ==============================================================

class PlantDiseaseCNN(nn.Module):
    """
    CNN de 9 capas basada en el paper de Geetharamani & Arun Pandian (2019).
    Adaptada para clasificar 10 clases de enfermedades de tomate.
    """
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()

        # Bloque convolucional: aplica filtro, normaliza, activa y reduce tamano
        def bloque_conv(canales_entrada, canales_salida):
            return nn.Sequential(
                # Conv2d: aplica filtros 3x3 para extraer caracteristicas
                # padding=1 mantiene el tamano espacial antes del pooling
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=3, padding=1),
                # BatchNorm: normaliza las activaciones para estabilizar el entrenamiento
                nn.BatchNorm2d(canales_salida),
                # ReLU: funcion de activacion no lineal (max(0, x))
                # Sin ella, la red seria equivalente a una transformacion lineal
                nn.ReLU(inplace=True),
                # MaxPool2d: reduce la resolucion a la mitad, conservando lo mas relevante
                nn.MaxPool2d(kernel_size=2, stride=2),
            )

        # 6 bloques convolucionales con filtros crecientes
        # Mas filtros = mas caracteristicas aprendidas en cada nivel
        self.features = nn.Sequential(
            bloque_conv(3, 32),      # Bloque 1: 128x128x3  -> 64x64x32
            bloque_conv(32, 64),     # Bloque 2: 64x64x32   -> 32x32x64
            bloque_conv(64, 128),    # Bloque 3: 32x32x64   -> 16x16x128
            bloque_conv(128, 256),   # Bloque 4: 16x16x128  -> 8x8x256
            bloque_conv(256, 512),   # Bloque 5: 8x8x256    -> 4x4x512
            bloque_conv(512, 512),   # Bloque 6: 4x4x512    -> 2x2x512
        )

        # Clasificador: capas densas que convierten caracteristicas en predicciones
        self.classifier = nn.Sequential(
            # Dropout: apaga aleatoriamente el 50% de las neuronas
            # Esto fuerza a la red a no depender de neuronas individuales (regularizacion)
            nn.Dropout(dropout),
            # Capa densa 1: 2048 features (512*2*2) -> 512
            nn.Linear(512 * 2 * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            # Capa de salida: 512 -> num_classes (10 enfermedades de tomate)
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        # Pasar por los bloques convolucionales
        x = self.features(x)
        # Aplanar: de (batch, 512, 2, 2) a (batch, 2048)
        x = x.view(x.size(0), -1)
        # Pasar por el clasificador
        x = self.classifier(x)
        return x


# Crear el modelo y moverlo a GPU
model_cnn = PlantDiseaseCNN(num_classes=num_classes).to(device)

# --- Resumen del modelo ---
print('=' * 60)
print('ARQUITECTURA: PlantDiseaseCNN (9 capas)')
print('=' * 60)
total_params = 0
for name, param in model_cnn.named_parameters():
    print(f'{name:45s} | forma: {str(list(param.shape)):20s} | params: {param.numel():>10,}')
    total_params += param.numel()
print('=' * 60)
print(f'Total de parametros: {total_params:,}')
print(f'Tamano estimado del modelo: {total_params * 4 / 1e6:.1f} MB (float32)')

---
## 5. Hora de entrenar — la computadora aprende a diagnosticar!

Ahora le vamos a mostrar miles de fotos al modelo, una y otra vez, para que aprenda a distinguir las enfermedades. El proceso funciona asi:

1. El modelo mira un grupo de fotos e intenta adivinar que enfermedad tiene cada una.
2. Comparamos sus respuestas con las respuestas correctas y calculamos "que tan mal le fue" (eso es la **loss** o perdida).
3. El modelo ajusta sus filtros internos para equivocarse menos la proxima vez (eso es el **backpropagation**).
4. Repetimos todo el proceso muchas veces. Cada pasada completa por todas las fotos se llama una **epoca**.

Es como estudiar para un examen: cada vez que repasas el material, entiendes un poco mejor. Vamos a hacer **30 repasos** (epocas).

Configuracion que usaremos (la misma del paper):
- **Optimizador Adam** — un algoritmo inteligente que ajusta la velocidad de aprendizaje automaticamente
- **Learning rate 0.001** — que tan grandes son los "pasos" de ajuste (muy grande = se pasa de largo, muy chico = tarda mucho)

Observa en las graficas como la accuracy (porcentaje de aciertos) va subiendo y la loss (errores) va bajando con cada epoca.

In [ ]:
# ==============================================================
# FUNCION DE ENTRENAMIENTO Y EVALUACION
# Reutilizable para ambos modelos (CNN y ResNet18)
# ==============================================================

def entrenar_modelo(modelo, train_loader, val_loader, num_epochs=30, lr=0.001,
                    nombre='Modelo'):
    """
    Entrena un modelo y registra metricas por epoca.

    Args:
        modelo: red neuronal (nn.Module)
        train_loader: DataLoader de entrenamiento
        val_loader: DataLoader de validacion
        num_epochs: numero de epocas de entrenamiento
        lr: learning rate para el optimizador Adam
        nombre: nombre del modelo (para los prints)

    Returns:
        dict con historiales de loss, accuracy y tiempo total
    """
    # CrossEntropyLoss: funcion de perdida para clasificacion multiclase
    # Combina LogSoftmax + NLLLoss en una sola operacion (mas estable numericamente)
    criterion = nn.CrossEntropyLoss()

    # Adam: optimizador adaptativo que ajusta el LR para cada parametro
    # Generalmente converge mas rapido que SGD
    optimizer = optim.Adam(modelo.parameters(), lr=lr)

    # Registrar metricas por epoca
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
    }

    print(f'\n{"=" * 60}')
    print(f'Entrenando: {nombre}')
    print(f'Epocas: {num_epochs} | LR: {lr} | Batch size: {BATCH_SIZE}')
    print(f'{"=" * 60}')

    inicio = time.time()
    mejor_val_acc = 0

    for epoch in range(num_epochs):
        # ============ FASE DE ENTRENAMIENTO ============
        modelo.train()  # activar dropout y batch norm en modo entrenamiento
        train_loss = 0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            # Mover datos a GPU
            images, labels = images.to(device), labels.to(device)

            # Forward pass: obtener predicciones del modelo
            outputs = modelo(images)

            # Calcular la perdida (que tan lejos esta de la respuesta correcta)
            loss = criterion(outputs, labels)

            # Backward pass: calcular gradientes
            optimizer.zero_grad()  # limpiar gradientes anteriores
            loss.backward()        # calcular gradientes por backpropagation
            optimizer.step()       # actualizar pesos del modelo

            # Acumular metricas
            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)  # clase con mayor probabilidad
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

        # ============ FASE DE VALIDACION ============
        modelo.eval()  # desactivar dropout; batch norm usa estadisticas globales
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():  # no calcular gradientes (ahorra memoria y tiempo)
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = modelo(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        # Calcular metricas promedio de la epoca
        train_loss_avg = train_loss / train_total
        val_loss_avg = val_loss / val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss_avg)
        history['val_loss'].append(val_loss_avg)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Guardar mejor modelo
        if val_acc > mejor_val_acc:
            mejor_val_acc = val_acc
            mejor_epoch = epoch + 1

        # Imprimir progreso cada 5 epocas (o la primera y ultima)
        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == num_epochs - 1:
            elapsed = time.time() - inicio
            print(f'Epoca {epoch+1:3d}/{num_epochs} | '
                  f'Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc:.4f} | '
                  f'Tiempo: {elapsed:.0f}s')

    tiempo_total = time.time() - inicio
    history['tiempo_total'] = tiempo_total
    history['mejor_val_acc'] = mejor_val_acc
    history['mejor_epoch'] = mejor_epoch

    print(f'\nEntrenamiento completado en {tiempo_total:.0f}s ({tiempo_total/60:.1f} min)')
    print(f'Mejor validacion accuracy: {mejor_val_acc:.4f} (epoca {mejor_epoch})')

    return history

In [ ]:
# ==============================================================
# ENTRENAR LA CNN DEL PAPER
# ==============================================================
NUM_EPOCHS = 30  # Ajustable: 30 epocas es suficiente para ver convergencia

history_cnn = entrenar_modelo(
    modelo=model_cnn,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    lr=0.001,
    nombre='CNN del paper (9 capas)'
)

# --- Graficar curvas de aprendizaje ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

# Accuracy
axes[0].plot(epochs_range, history_cnn['train_acc'], 'b-', label='Train', linewidth=2)
axes[0].plot(epochs_range, history_cnn['val_acc'], 'g-', label='Validacion', linewidth=2)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CNN del paper: Accuracy', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Loss
axes[1].plot(epochs_range, history_cnn['train_loss'], 'b-', label='Train', linewidth=2)
axes[1].plot(epochs_range, history_cnn['val_loss'], 'g-', label='Validacion', linewidth=2)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('CNN del paper: Loss', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nAccuracy final de validacion: {history_cnn["val_acc"][-1]:.2%}')

---
## 6. El atajo inteligente: Transfer Learning con ResNet18

Nuestro primer modelo aprendio todo desde cero: empezo sin saber nada y tuvo que descubrir por si solo como detectar bordes, texturas y formas. Funciona, pero... y si pudieramos empezar con ventaja?

Eso es exactamente lo que hace el **Transfer Learning** (aprendizaje por transferencia). La idea es brillante:

> Tomar un modelo que ya aprendio a "ver" con millones de imagenes y solo ensenarle la parte nueva: "estas son las enfermedades del tomate".

Es como si en vez de ensenarle a un bebe a reconocer formas desde cero, tomaras a un fotografo experto y solo le dijeras: "mira, estas son las 10 enfermedades que necesito que identifiques".

Usaremos **ResNet18**, un modelo que fue entrenado con **1.2 millones de fotos** del proyecto ImageNet (fotos de perros, autos, flores, edificios... de todo). Sus primeras capas ya saben detectar bordes, texturas y formas complejas. Nosotros solo reemplazamos la ultima capa (que antes distinguia 1000 categorias de objetos) por una nueva que distingue nuestras 10 enfermedades.

Dato curioso: "congelamos" todas las capas existentes (no las modificamos) y solo entrenamos la ultima. Eso significa que de los ~11 millones de parametros del modelo, solo vamos a entrenar unos pocos miles!

In [ ]:
# ==============================================================
# MODELO RESNET18 CON TRANSFER LEARNING
# ==============================================================

# Cargar ResNet18 preentrenada en ImageNet (1.2M imagenes, 1000 clases)
model_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congelar todas las capas base
# Esto evita que se modifiquen los filtros ya aprendidos
# Solo entrenaremos la ultima capa (clasificador)
for param in model_resnet.parameters():
    param.requires_grad = False

# Reemplazar la capa final: de 512->1000 (ImageNet) a 512->10 (nuestras clases)
# Esta nueva capa SI se entrena (requires_grad=True por defecto)
num_features = model_resnet.fc.in_features  # 512 en ResNet18
model_resnet.fc = nn.Linear(num_features, num_classes)

model_resnet = model_resnet.to(device)

# Contar parametros
total_params_resnet = sum(p.numel() for p in model_resnet.parameters())
trainable_params = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
frozen_params = total_params_resnet - trainable_params

print('=' * 60)
print('ARQUITECTURA: ResNet18 (Transfer Learning)')
print('=' * 60)
print(f'Parametros totales:      {total_params_resnet:>10,}')
print(f'Parametros congelados:   {frozen_params:>10,}  (capas preentrenadas)')
print(f'Parametros entrenables:  {trainable_params:>10,}  (solo el clasificador)')
print(f'\nComparacion con CNN del paper: {total_params_resnet/sum(p.numel() for p in model_cnn.parameters()):.1f}x parametros totales')
print(f'Pero solo entrenamos {trainable_params/sum(p.numel() for p in model_cnn.parameters()):.1%} de lo que entrena la CNN')

In [ ]:
# ==============================================================
# ENTRENAR RESNET18
# Mismo numero de epocas para comparacion justa
# ==============================================================

history_resnet = entrenar_modelo(
    modelo=model_resnet,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    lr=0.001,
    nombre='ResNet18 (Transfer Learning)'
)

# --- Graficar curvas de aprendizaje ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, history_resnet['train_acc'], 'b-', label='Train', linewidth=2)
axes[0].plot(epochs_range, history_resnet['val_acc'], 'g-', label='Validacion', linewidth=2)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('ResNet18: Accuracy', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

axes[1].plot(epochs_range, history_resnet['train_loss'], 'b-', label='Train', linewidth=2)
axes[1].plot(epochs_range, history_resnet['val_loss'], 'g-', label='Validacion', linewidth=2)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('ResNet18: Loss', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nAccuracy final de validacion: {history_resnet["val_acc"][-1]:.2%}')

---
## 7. El momento de la verdad: quien aprendio mejor?

Tenemos dos modelos entrenados con las mismas fotos y las mismas condiciones. Ahora vamos a ponerlos frente a frente y ver cual diagnostica mejor las enfermedades.

Vamos a comparar:
- **Accuracy** — que porcentaje de hojas clasifico correctamente?
- **Tiempo** — cuanto tardo en aprender?
- **Parametros** — que tan "grande" es cada modelo?

Ademas veremos la **matriz de confusion**, que nos dice exactamente donde se equivoca el modelo: confunde el tizon temprano con el tardio? Le cuesta reconocer el virus del mosaico? Esa informacion es oro para mejorar el sistema.

In [ ]:
# ==============================================================
# TABLA COMPARATIVA
# ==============================================================

cnn_params = sum(p.numel() for p in model_cnn.parameters())
resnet_params = sum(p.numel() for p in model_resnet.parameters())
resnet_trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)

print('=' * 70)
print(f'{"Metrica":<30s} {"CNN del paper":>18s} {"ResNet18 (TL)":>18s}')
print('=' * 70)
print(f'{"Mejor Val Accuracy":<30s} {history_cnn["mejor_val_acc"]:>17.2%} {history_resnet["mejor_val_acc"]:>17.2%}')
print(f'{"Val Accuracy (ultima epoca)":<30s} {history_cnn["val_acc"][-1]:>17.2%} {history_resnet["val_acc"][-1]:>17.2%}')
print(f'{"Tiempo de entrenamiento":<30s} {history_cnn["tiempo_total"]:>15.0f} s {history_resnet["tiempo_total"]:>15.0f} s')
print(f'{"Parametros totales":<30s} {cnn_params:>18,} {resnet_params:>18,}')
print(f'{"Parametros entrenados":<30s} {cnn_params:>18,} {resnet_trainable:>18,}')
print('=' * 70)

# --- Graficas lado a lado ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

# Accuracy de validacion
axes[0].plot(epochs_range, history_cnn['val_acc'], 'b-o', label='CNN del paper',
             linewidth=2, markersize=3)
axes[0].plot(epochs_range, history_resnet['val_acc'], 'r-s', label='ResNet18 (TL)',
             linewidth=2, markersize=3)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Comparacion: Val Accuracy', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Loss de validacion
axes[1].plot(epochs_range, history_cnn['val_loss'], 'b-o', label='CNN del paper',
             linewidth=2, markersize=3)
axes[1].plot(epochs_range, history_resnet['val_loss'], 'r-s', label='ResNet18 (TL)',
             linewidth=2, markersize=3)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Comparacion: Val Loss', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================
# MATRIZ DE CONFUSION DEL MEJOR MODELO
# ==============================================================

# Determinar cual modelo tuvo mejor accuracy
if history_resnet['mejor_val_acc'] >= history_cnn['mejor_val_acc']:
    mejor_modelo = model_resnet
    mejor_nombre = 'ResNet18 (Transfer Learning)'
else:
    mejor_modelo = model_cnn
    mejor_nombre = 'CNN del paper'

print(f'Mejor modelo: {mejor_nombre}')
print('Generando matriz de confusion...\n')

# Obtener predicciones en el conjunto de validacion
mejor_modelo.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = mejor_modelo(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Nombres cortos para las clases
short_names = [n.replace('Tomato___', '').replace('Tomato_', '').replace('_', ' ')
               for n in class_names]

# Calcular matriz de confusion
cm = confusion_matrix(all_labels, all_preds)

# Graficar
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title(f'Matriz de Confusion: {mejor_nombre}', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax)

# Etiquetas
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Prediccion', fontsize=12)
ax.set_ylabel('Clase real', fontsize=12)

# Numeros dentro de la matriz
thresh = cm.max() / 2
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=8)

plt.tight_layout()
plt.show()

# Reporte de clasificacion
print('\nReporte de clasificacion:')
print(classification_report(all_labels, all_preds, target_names=short_names))

In [ ]:
# ==============================================================
# CLASIFICACION DE EJEMPLO CON IMAGENES REALES
# Seleccionamos imagenes aleatorias del set de validacion
# y mostramos la prediccion del mejor modelo
# ==============================================================

# Funcion para desnormalizar imagenes (revertir la normalizacion para visualizar)
def desnormalizar(tensor):
    """Revierte la normalizacion de ImageNet para visualizar la imagen."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

# Obtener un batch de validacion
images, labels = next(iter(val_loader))
images_gpu = images.to(device)

mejor_modelo.eval()
with torch.no_grad():
    outputs = mejor_modelo(images_gpu)
    probs = torch.softmax(outputs, dim=1)
    _, predicted = outputs.max(1)

# Mostrar 8 ejemplos aleatorios
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
indices = random.sample(range(len(images)), 8)

for ax, idx in zip(axes.flatten(), indices):
    img = desnormalizar(images[idx])
    pred = predicted[idx].item()
    real = labels[idx].item()
    prob = probs[idx, pred].item()

    ax.imshow(img)
    pred_name = short_names[pred]
    real_name = short_names[real]

    color = 'green' if pred == real else 'red'
    ax.set_title(f'Pred: {pred_name}\nReal: {real_name}\nConf: {prob:.1%}',
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

fig.suptitle(f'Predicciones del {mejor_nombre}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Verde = prediccion correcta | Rojo = prediccion incorrecta')

---
## 8. Que aprendimos y hacia donde ir?

### Lo que logramos

Acabamos de hacer algo que hasta hace pocos anos requeria equipos de investigacion completos: **entrenar una inteligencia artificial que diagnostica enfermedades en plantas con mas del 90% de precision**. Y lo hicimos en menos de una hora, desde un navegador web, gratis.

Entrenamos dos modelos y los comparamos:

1. **CNN desde cero** — Construimos la arquitectura pieza por pieza, como armar un LEGO siguiendo las instrucciones del paper. El modelo tuvo que aprender TODO: desde que es un borde hasta que es una mancha de tizon.

2. **ResNet18 con Transfer Learning** — Tomamos un modelo que ya "sabia ver" y le ensenamos nuestra tarea especifica. Como contratar a un experto y darle una capacitacion corta.

### Que enfoque es mejor?

No hay una respuesta unica. Depende de la situacion:

| | CNN desde cero | Transfer Learning |
|---|---|---|
| **Cuando usarlo** | Cuando tu problema es muy diferente a fotos naturales, o cuando necesitas una red pequena y eficiente | Cuando tienes pocas fotos, poco tiempo, o tu problema se parece a lo que ImageNet ya conoce |
| **Ventaja principal** | Control total: tu decides cada detalle | Resultados buenos rapido, con menos datos |
| **Desventaja** | Necesita mas datos y mas tiempo para alcanzar buen rendimiento | Dependes de que el modelo base sea compatible con tu problema |

### Ideas para seguir investigando

Si este tema te atrapo, aqui hay algunas preguntas abiertas que podrias explorar:

- **Data augmentation:** Que pasa si artificialmente creamos mas fotos rotando, volteando o cambiando el brillo de las existentes? Mejora el modelo?
- **Mas clases:** Podrias expandir esto a las 39 clases del dataset completo (todas las plantas, no solo tomate)?
- **En el campo real:** Las fotos de PlantVillage tienen fondo uniforme. Como se comportaria el modelo con fotos tomadas con un celular en un cultivo real? Ese es uno de los grandes desafios.
- **Modelos mas modernos:** Que pasa si en vez de ResNet18 usamos EfficientNet, o un Vision Transformer (ViT)?
- **Aplicacion movil:** Podrias exportar este modelo para que funcione en una app de celular? (pista: investiga ONNX o TorchScript).

### El panorama mas grande

Lo que hicimos aqui es un ejemplo de como la inteligencia artificial puede tener **impacto real en la sociedad**. Las enfermedades en cultivos causan perdidas de hasta el 30% de la produccion mundial de alimentos. Un agricultor con un celular y un modelo como este podria diagnosticar problemas dias antes de que sean visibles a simple vista.

Ya hay startups y proyectos de investigacion trabajando en esto. Tal vez el proximo avance lo hagas tu.

---

**Referencia del paper:** Geetharamani, G., & Arun Pandian, J. (2019). *Identification of plant leaf diseases using a nine-layer deep convolutional neural network.* Computers & Electrical Engineering, 76, 323-338.

**Dataset:** PlantVillage — disponible libremente en Kaggle para investigacion.